# DentaBridge: tuning Qwen3-VL-4B on the free Kaggle T4s

Settings: **Accelerator** = GPU T4 x2, **Internet** = On. Add the private dataset `dentabridge-code`
(built by `make kaggle-bundle`) as an input. Set `STAGES` below: the default only installs, loads the
data and runs ten training steps, and prints how long the full run would take. Then add
`train`, `merge`, `eval` and `zeroshot` and run again (a session lasts at most 12 hours; the weekly
GPU quota is 30 hours and counts the session, not the number of GPUs).

With two GPUs the training runs as two lanes side by side: **A** tunes the language layers
(`lean-t4`), **B** the vision tower as well (`lean-t4-vision`). With one GPU only lane A runs.
The same stages run on Colab with `WORK=/content SCRATCH=/content/scratch ROOT=/content/dentabridge`.

In [ ]:
STAGES = ["setup", "data", "smoke"]  # + "train", "merge", "eval", "zeroshot" for the full run

In [ ]:
import os
import pathlib
import shutil
import subprocess

ROOT = pathlib.Path("/kaggle/working/dentabridge")
if not ROOT.exists():
    inputs = pathlib.Path("/kaggle/input")
    archive = next(inputs.rglob("dentabridge-code.tar.gz"), None)
    if archive:
        subprocess.run(["tar", "-xzf", str(archive), "-C", "/kaggle/working"], check=True)
    else:  # Kaggle unpacks an uploaded archive into a folder
        folder = next(p.parent for p in inputs.rglob("COMMIT") if p.parent.name == "dentabridge")
        shutil.copytree(folder, ROOT, ignore=shutil.ignore_patterns("._*"))
print("code at commit", (ROOT / "COMMIT").read_text().strip())


def run_stage(name, tag="", **env):
    if name not in STAGES:
        print(f"{tag}skipped:", name)
        return
    process = subprocess.Popen(
        ["bash", str(ROOT / "scripts/kaggle/run.sh"), name],
        env={**os.environ, **env},
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )
    for line in process.stdout:
        print(tag + line, end="", flush=True)
    if process.wait():
        raise RuntimeError(f"{tag}stage {name} failed with exit code {process.returncode}")

## 1. Install

In [ ]:
run_stage("setup")

## 2. Data (AKU and AlphaDent from Zenodo, CC BY 4.0)

In [ ]:
run_stage("data")

## 3. Smoke test: ten steps, and the estimate for the full run

In [ ]:
run_stage("smoke")

## 4. Train, merge and score on the validation split
One lane per GPU, side by side; lane A also scores the untuned model with the same prompt (the
baseline). A lane that fails does not stop the other. A `checkpoint-*` folder lets a lane resume.

In [ ]:
LANES = {
    "A": {"gpu": 0, "config": "lean-t4", "stages": ["train", "merge", "eval", "zeroshot"]},
    "B": {"gpu": 1, "config": "lean-t4-vision", "stages": ["train", "merge", "eval"]},
}

In [ ]:
import threading

gpus = len(subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout.splitlines())
lanes = {tag: lane for tag, lane in LANES.items() if lane["gpu"] < gpus}
skipped = sorted(set(LANES) - set(lanes)) or "none"
print(f"{gpus} GPU(s): lanes {', '.join(lanes)}; skipped for lack of a GPU: {skipped}")
failures = []


def run_lane(tag, lane):
    try:
        for name in lane["stages"]:
            run_stage(name, tag=f"[{tag}] ", GPU=str(lane["gpu"]), CONFIG=lane["config"])
    except Exception as error:  # the other lane carries on; the notebook fails at the end
        failures.append(f"{tag}: {error}")


threads = [threading.Thread(target=run_lane, args=item) for item in lanes.items()]
for thread in threads:
    thread.start()
for thread in threads:
    thread.join()
if failures:
    raise RuntimeError("; ".join(failures))

## 5. Results

In [ ]:
import glob
import pathlib

for path in sorted(glob.glob("/kaggle/working/out/*/val/val.md")):
    print(pathlib.Path(path).read_text())

Download from the **Output** tab: `out/lean-t4/adapter` and `out/lean-t4-vision/adapter`, every
`run.json` and `out/*/val/`.